# Ponowne profilowanie accDM — 30 punktów

Notebook uruchamia nowy profiler na **zapisanych widmach i istniejących danych projektu**. Domyślnie wykonuje czteropunktowy pilot. Ustawienia `PHASE = "campaign"` i `RESUME = True` uruchamiają pełną kampanię po pomyślnym pilocie.

Pakiet zawiera indeks danych oraz kod; produkcyjne NPZ i fit JSON muszą być dostępne w Twoim `lyalpha_one_loop`. Testy lokalne pakietu nie są nowymi wynikami accDM. Szczegóły metod, ograniczeń i walidacji: **README.md**.

Pracuj w środowisku, w którym działa dotychczasowy fitter. Zachowaj ten notebook razem z modułami pakietu. Nie uruchamiaj równocześnie drugiego procesu z tym samym katalogiem wyników.


In [1]:
from pathlib import Path
import sys
import json
import subprocess
import zipfile
import pandas as pd
from IPython.display import display, Image, FileLink

# Notebook otwarty w rozpakowanym katalogu albo w katalogu projektu.
PACKAGE_DIR = Path.cwd()
if not (PACKAGE_DIR / "run_reprofile.py").is_file():
    PACKAGE_DIR = PACKAGE_DIR / "accdm_reprofile_30_points"
PACKAGE_DIR = PACKAGE_DIR.resolve()

roots = [PACKAGE_DIR.parent,
         Path("/home/2/ks405818/Master/lyalpha_one_loop"),
         Path("/home/krzysztof/Workspace/lyalpha_one_loop")]
PROJECT_ROOT = next((p for p in roots if (p / "lyalpha_pt/fit.py").is_file()), roots[1])
# W razie potrzeby wpisz tutaj własną ścieżkę:
# PROJECT_ROOT = Path("/pełna/ścieżka/lyalpha_one_loop")
OUTPUT_DIR = PROJECT_ROOT / "runs/accdm/reprofile_30_v1"

PHASE = "pilot"        # pilot | campaign | validate | report
RESUME = False         # True przy wznowieniu albo przejściu pilot -> campaign
LOCAL_BUDGET = 5000
GLOBAL_BUDGET = 20000
SEEDS = (12345, 23456)

print("Pakiet:", PACKAGE_DIR)
print("Projekt:", PROJECT_ROOT)
print("Wyniki:", OUTPUT_DIR)
print("Faza:", PHASE, "| wznowienie:", RESUME)


Pakiet: /home/2/ks405818/Master/lyalpha_one_loop/accdm_reprofile_30_points
Projekt: /home/2/ks405818/Master/lyalpha_one_loop
Wyniki: /home/2/ks405818/Master/lyalpha_one_loop/runs/accdm/reprofile_30_v1
Faza: pilot | wznowienie: False


## Wybrane punkty

22 punkty obejmują główne oscylacje, trzy kontrolują znaną poprawę, cztery leżą przy interesującym konturze, jeden stanowi kontrolę małego f. Brakujące punkty nie będą interpolowane. Pilot uruchomi tylko cztery wskazane poniżej; sam manifest obejmuje całe 30.


In [2]:
assert (PACKAGE_DIR / "run_reprofile.py").is_file(), "Ustaw poprawny PACKAGE_DIR."
manifest = pd.read_csv(PACKAGE_DIR / "pilot_points.csv")
assert len(manifest) == 30 and manifest.coordinate_key.is_unique
pilot_keys = {"14.7142857143|-1.3750000000", "15.0000000000|-1.3750000000",
              "19.2857142857|-4.0000000000", "14.7142857143|-0.1000000000"}
display(manifest.assign(pilot=manifest.coordinate_key.isin(pilot_keys)))


,coordinate_key,log10m_acc,log10f_acc,groups,role,pilot
0,16.0000000000|-0.3000000000,16.000000,-0.300,horizontal_f_m0p3,main_oscillation,False
1,16.1428571429|-0.3000000000,16.142857,-0.300,horizontal_f_m0p3,main_oscillation,False
2,16.2857142857|-0.3000000000,16.285714,-0.300,horizontal_f_m0p3,main_oscillation,False
3,16.4285714286|-0.3000000000,16.428571,-0.300,horizontal_f_m0p3;vertical_m16p4285714286,main_oscillation,False
4,16.5714285714|-0.3000000000,16.571429,-0.300,horizontal_f_m0p3,main_oscillation,False
5,16.7142857143|-0.3000000000,16.714286,-0.300,horizontal_f_m0p3,main_oscillation,False
6,16.8571428571|-0.3000000000,16.857143,-0.300,horizontal_f_m0p3,main_oscillation,False
7,16.0000000000|0.0000000000,16.000000,0.000,horizontal_f_0,main_oscillation,False
8,16.1428571429|0.0000000000,16.142857,0.000,horizontal_f_0,main_oscillation,False
9,16.2857142857|0.0000000000,16.285714,0.000,horizontal_f_0,main_oscillation,False


## Uruchomienie

Komórka poniżej wykonuje wybraną fazę i pokazuje bieżący postęp. Domyślny pilot jest kontrolą odtworzenia modelu oraz odzyskania znanych poprawek; nie dowodzi znalezienia globalnych minimów. Dopiero po jego powodzeniu można ustawić `PHASE = "campaign"`, `RESUME = True` i wykonać notebook ponownie.

Po przerwaniu zadania ustaw `RESUME = True`, zachowując pozostałe ustawienia. Przy zmianie kodu, budżetów, danych albo manifestu wybierz nowy katalog wyników. Zakończone próby mają punkty kontrolne; przerwana próba zostanie powtórzona.


In [3]:
assert PHASE in {"pilot", "campaign", "validate", "report"}
if PHASE != "report":
    assert (PROJECT_ROOT / "lyalpha_pt/fit.py").is_file(), "Nie znaleziono fittera — ustaw PROJECT_ROOT."
    if (OUTPUT_DIR / "state.json").exists() and not RESUME:
        raise RuntimeError("Ten katalog ma zapisany stan. Ustaw RESUME=True albo wybierz nowy OUTPUT_DIR.")

command = [sys.executable, str(PACKAGE_DIR / "run_reprofile.py"),
           "--project-root", str(PROJECT_ROOT), "--out", str(OUTPUT_DIR),
           "--phase", PHASE, "--local-budget", str(LOCAL_BUDGET),
           "--global-budget", str(GLOBAL_BUDGET), "--seeds", *map(str, SEEDS)]
if RESUME:
    command.append("--resume")
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1, cwd=PACKAGE_DIR)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("Przerwano. Zakończone próby pozostają zapisane; wznowienie: RESUME=True.")
    raise
if returncode:
    error_file = OUTPUT_DIR / "last_error.json"
    if error_file.is_file():
        print(error_file.read_text())
    raise RuntimeError(f"Proces zakończył się kodem {returncode}. Sprawdź komunikat powyżej.")


RuntimeError: Ten katalog ma zapisany stan. Ustaw RESUME=True albo wybierz nowy OUTPUT_DIR.

## Poprawa dopasowania i status prób

`gain` oznacza spadek χ² najlepszego jawnie ocenionego wektora. `budget_exhausted` oznacza brak potwierdzenia zbieżności tej próby, nawet jeśli znaleziono poprawę. `stability_assessment` wymaga zgodności udanych końcowych wyników obu kierunków i metody sześciowymiarowej; zachowany stary wynik nie zastępuje takiego testu.


In [ ]:
points_path = OUTPUT_DIR / "reprofile_best_points.csv"
points = pd.read_csv(points_path)
columns = ["target", "chi2_old", "chi2_best", "gain", "delta_chi2_best",
           "optimizer_successes", "budget_exhausted", "stability_assessment"]
display(points[[c for c in columns if c in points]].sort_values("gain", ascending=False))

gate_path = OUTPUT_DIR / "pilot_gate.json"
if gate_path.is_file():
    gate = json.loads(gate_path.read_text())
    print("Bramka pilota:", gate["passed"], gate["checks"])
attempts_path = OUTPUT_DIR / "reprofile_attempts.csv"
try:
    attempts = pd.read_csv(attempts_path)
except pd.errors.EmptyDataError:
    attempts = pd.DataFrame()
if len(attempts):
    display(attempts.groupby(["method", "termination"]).size().rename("liczba_prób").reset_index())


## Wykresy przed i po

Wykresy mają wspólną referencję χ² z pełnego snapshotu. Braki pozostają przerwami. Po pilocie większość przekrojów będzie jeszcze pusta — ich wypełnienie wymaga fazy `campaign`. Pełny zestaw parametrów nuisance, odległości od granic i P1D jest w `reports/`.


In [ ]:
for filename in ["reprofile_chi2_cuts.png", "reprofile_before_after.png",
                 "reprofile_direction_comparison.png"]:
    path = OUTPUT_DIR / "reports" / filename
    if path.is_file():
        display(Image(filename=str(path)))
status_path = OUTPUT_DIR / "reports/report_status.json"
if status_path.is_file():
    status = json.loads(status_path.read_text())
    for warning in status.get("warnings", []):
        print("Informacja raportu:", warning)


## Paczka wyników do dalszej analizy

Poniższa komórka zbiera tabele, metadane, stan oraz wykresy z bieżącej fazy. Przekaż utworzony ZIP. W pierwszej kolejności ocenimy znane poprawki, główne przekroje χ², zgodność metod i aktywne granice. Jeśli oscylacje pozostaną mimo zgodnych minimów, wybierzemy kilka sąsiadów do osobnego testu dokładności widm/projekcji.


In [ ]:
result_zip = OUTPUT_DIR.parent / (OUTPUT_DIR.name + "_wyniki.zip")
with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file() and path.suffix.lower() in {".csv", ".json", ".png"}:
            archive.write(path, arcname=OUTPUT_DIR.name + "/" + str(path.relative_to(OUTPUT_DIR)))
print("Zapisano:", result_zip)
display(FileLink(str(result_zip)))
